# Arenda.az Data Collection & Preparation Report

This notebook builds a property dataset for multimodal modeling.

## Workflow at a glance
1. Collect listing URLs
2. Scrape tabular property features
3. Download listing images
4. Merge and clean all batch outputs

In [2]:
from tqdm import tqdm
import pandas as pd
import requests
from bs4 import BeautifulSoup
import regex as re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.chrome.service import Service
import urllib.request
import numpy as np
from selenium.webdriver.chrome.options import Options
import os
from pathlib import Path
from urllib.parse import urlparse

## 1) Environment Setup
Load required scraping, parsing, and utility libraries.

In [ ]:
# Setup Chrome options

chrome_options = Options()
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36')

# Use ChromeDriverManager
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# Anti-detection
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

url = 'https://arenda.az/filtirli-axtaris/?home_search=1&lang=1&site=1&home_s=1&elan_novu%5B1%5D=1&emlak_novu%5B370%5D=370&emlak_novu%5B371%5D=371&emlak_novu%5B5%5D=5&emlak_novu%5B861%5D=861&price_min=&price_max=&axtar=&sahe_min=&sahe_max=&mertebe_min=&mertebe_max=&y_mertebe_min=&y_mertebe_max='
driver.get(url)

time.sleep(3)

## 2) Listing URL Discovery
Open filtered search pages, parse HTML, and collect unique `satilir` listing links.

**Output:** `satilir_links.csv`

In [3]:
# Get page source from Selenium driver instead of requests
page_source = driver.page_source
bs = BeautifulSoup(page_source, 'html.parser')

In [4]:
bs

<html class="html_az" id="html" lang="az"><head><script async="" src="https://st.top100.ru/top100/3.18.16/mgc.js" type="text/javascript"></script><script async="" src="https://www.google-analytics.com/analytics.js" type="text/javascript"></script><script async="" src="https://www.googletagmanager.com/gtag/js?id=G-EXGHL2Q2LY&amp;cx=c&amp;_slc=1" type="text/javascript"></script><script async="" src="https://st.top100.ru/top100/top100.js" type="text/javascript"></script><script async="" src="//www.google-analytics.com/analytics.js"></script> <meta charset="utf-8"/> <link href="https://arenda.az/public/css/img/favicon.png" rel="shortcut icon"/> <link href="https://arenda.az/public/css/img/favicon.png" rel="apple-touch-icon-precomposed"/> <meta content="37fb16dcf3d065c2" name="yandex-verification"/> <!-- Title --> <title> Ətraflı axtarış </title> <link crossorigin="anonymous" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.7.2/css/all.min.css" integrity="sha512-Evv84Mr4kqVGRNSgI

In [5]:
# Extract all href links
links = []
for a_tag in bs.find_all('a', href=True):
    links.append(a_tag['href'])

# Display unique links
print(f"Total links found: {len(links)}")
print(f"Unique links: {len(set(links))}")
print("\nFirst 10 links:")
for link in links:
    print(link)

Total links found: 143
Unique links: 120

First 10 links:
https://www.facebook.com/arenda.az
https://www.instagram.com/arenda.az/
https://www.youtube.com/channel/UCZFYL3miTL_Eq9jBGfEw_qQ
https://www.tiktok.com/@arenda.az
https://wa.me/+994705962424
https://arenda.az/ru/rasshirennyy-poisk/?home_search=1&lang=1&site=1&home_s=1&elan_novu%5B1%5D=1&emlak_novu%5B370%5D=370&emlak_novu%5B371%5D=371&emlak_novu%5B5%5D=5&emlak_novu%5B861%5D=861&price_min=&price_max=&axtar=&sahe_min=&sahe_max=&mertebe_min=&mertebe_max=&y_mertebe_min=&y_mertebe_max=
https://arenda.az/xeberler
https://arenda.az/dasinmaz-emlak-agentlikleri
https://arenda.az/yasayis-kompleksi
https://arenda.az/istifadecilerin-reytinqi
tel:+994 70 596 24 24
https://arenda.az/
https://arenda.az/secilmisler
#login_box
#register_box
https://arenda.az/elan-yerlesdir
https://yigim.az/
https://yigim.az/
#premium_et
https://arenda.az/satilir-3-otaqli-yeni-tikili-bineqedi-rayonu-nesimi-metrosu-9-cu-mikrorayon-mir-celal-137a-4
https://arenda.az

In [6]:
# Filter links starting with "satilir"
satilir_links = [link for link in links if link.startswith('satilir') or '/satilir' in link]

print(f"Links with 'satilir': {len(satilir_links)}")
for link in satilir_links:
    # Make absolute URL if needed
    if link.startswith('/'):
        full_link = 'https://arenda.az' + link
    elif not link.startswith('http'):
        full_link = 'https://arenda.az/' + link
    else:
        full_link = link
    print(full_link)

Links with 'satilir': 30
https://arenda.az/satilir-3-otaqli-yeni-tikili-bineqedi-rayonu-nesimi-metrosu-9-cu-mikrorayon-mir-celal-137a-4
https://arenda.az/satilir-5-otaqli-yeni-tikili-xetai-rayonu-xetai-metrosu-baki-seheri-xetai-rayonu
https://arenda.az/satilir-4-otaqli-heyet-evi-villa-abseron-rayonu-masazir-eliaga-vahid-kuc-595
https://arenda.az/satilir-3-otaqli-yeni-tikili-nerimanov-rayonu-neriman-nerimanov-metrosu-montin-qes-nerimanov-rayonu-mayakovski-kucesi
https://arenda.az/satilir-1-otaqli-yeni-tikili-abseron-rayonu-masazir-azadliq-pr-60
https://arenda.az/satilir-2-otaqli-yeni-tikili-abseron-rayonu-masazir-abs-eron-rayon-masazir-qesebesi-153
https://arenda.az/satilir-2-otaqli-heyet-evi-villa-sabuncu-rayonu-koroglu-metrosu-mastaga-qes-sabuncu-rayonu-mastaga-qesebesi-5
https://arenda.az/satilir-5-otaqli-heyet-evi-villa-nizami-rayonu-xetai-metrosu-kesle-qes-nizami-rayonu-kesle-qesebesi-qasim-bey-zakir-kucesi-4-1
https://arenda.az/satilir-2-otaqli-yeni-tikili-abseron-rayonu-masazir-m

In [7]:
# Loop through all pages (1 to 296) and collect satilir links
all_satilir_links = []

base_url = 'https://arenda.az/filtirli-axtaris/{}/?home_search=1&lang=1&site=1&home_s=1&elan_novu%5B1%5D=1&emlak_novu%5B370%5D=370&emlak_novu%5B371%5D=371&emlak_novu%5B5%5D=5&emlak_novu%5B861%5D=861&price_min=&price_max=&axtar=&sahe_min=&sahe_max=&mertebe_min=&mertebe_max=&y_mertebe_min=&y_mertebe_max='

for page_num in tqdm(range(1, 297)):  # Pages 1 to 296
    try:
        # Navigate to page
        page_url = base_url.format(page_num)
        driver.get(page_url)
        time.sleep(2)  # Wait for page to load
        
        # Parse with BeautifulSoup
        page_source = driver.page_source
        bs = BeautifulSoup(page_source, 'html.parser')
        
        # Extract all links
        page_links = []
        for a_tag in bs.find_all('a', href=True):
            href = a_tag['href']
            page_links.append(href)
        
        # Filter satilir links
        satilir_links_page = [link for link in page_links if 'satilir' in link]
        
        # Convert to absolute URLs
        for link in satilir_links_page:
            if link.startswith('/'):
                full_link = 'https://arenda.az' + link
            elif not link.startswith('http'):
                full_link = 'https://arenda.az/' + link
            else:
                full_link = link
            
            # Avoid duplicates
            if full_link not in all_satilir_links:
                all_satilir_links.append(full_link)
        
        print(f"Page {page_num}: Found {len(satilir_links_page)} satilir links (Total: {len(all_satilir_links)})")
        
    except Exception as e:
        print(f"Error on page {page_num}: {e}")
        continue

print(f"\n✓ Scraping complete!")
print(f"Total unique satilir links collected: {len(all_satilir_links)}")

  0%|          | 0/296 [00:00<?, ?it/s]

  0%|          | 1/296 [00:04<23:00,  4.68s/it]

Page 1: Found 32 satilir links (Total: 32)


  1%|          | 2/296 [00:08<21:34,  4.40s/it]

Page 2: Found 31 satilir links (Total: 54)


...

Page 294: Found 41 satilir links (Total: 6015)


100%|█████████▉| 295/296 [21:53<00:04,  4.67s/it]

Page 295: Found 41 satilir links (Total: 6015)


100%|██████████| 296/296 [21:58<00:00,  4.45s/it]

Page 296: Found 41 satilir links (Total: 6015)

✓ Scraping complete!
Total unique satilir links collected: 6015


In [ ]:
# Save links to CSV and display sample
df_links = pd.DataFrame({'url': all_satilir_links})
df_links.to_csv('satilir_links.csv', index=False)

In [ ]:
# Load the saved links
df_links = pd.read_csv('satilir_links.csv')
print(f"Loaded {len(df_links)} links to scrape")
print(df_links.head())

Loaded 6297 links to scrape
                                                 url
0  https://arenda.az/satilir-4-otaqli-heyet-evi-v...
1  https://arenda.az/satilir-3-otaqli-yeni-tikili...
2  https://arenda.az/satilir-2-otaqli-yeni-tikili...
3  https://arenda.az/satilir-7-otaqli-bag-evi-bin...
4  https://arenda.az/satilir-3-otaqli-yeni-tikili...


In [8]:
for i in range(len(df_links)):
    url = df_links.loc[i, 'url']
    print(f"{i+1}/{len(df_links)}: {url}")

1/6297: https://arenda.az/satilir-4-otaqli-heyet-evi-villa-abseron-rayonu-masazir-azer-city-kuc
2/6297: https://arenda.az/satilir-3-otaqli-yeni-tikili-xetai-rayonu-hezi-aslanov-metrosu-kohne-gunesli-qes-general-sixlinski-kucesi-4
3/6297: https://arenda.az/satilir-2-otaqli-yeni-tikili-yasamal-rayonu-insaatcilar-metrosu-yeni-yasamal-qes-yeni-yasamal-qesebesi-mehemmed-xiyabani-kucesi-62
4/6297: https://arenda.az/satilir-7-otaqli-bag-evi-bineqedi-rayonu-avtovagzal-metrosu-bileceri-qes-baki-seheri-bineqedi-rayonu
5/6297: https://arenda.az/satilir-3-otaqli-yeni-tikili-abseron-rayonu-masazir-baki-seheri-abseron-rayonu-4
6/6297: https://arenda.az/satilir-1-otaqli-yeni-tikili-abseron-rayonu-masazir-azadliq-pr-60
7/6297: https://arenda.az/satilir-5-otaqli-heyet-evi-villa-xezer-rayonu-merdekan-28-may-2
8/6297: https://arenda.az/satilir-2-otaqli-yeni-tikili-sabuncu-rayonu-bakixanov-qes-genclik-kuc-29-c-m-94
9/6297: https://arenda.az/satilir-3-otaqli-heyet-evi-villa-sabuncu-rayonu-zabrat-qes-sabunc

## 3) Property Feature Extraction
Define helper functions and scrape base fields + dynamic amenities from each listing page.

**Output:** `satilir_properties_dynamic1.csv`

In [ ]:
# Iterate through saved links and extract base + dynamic features (Yes/No)

def _clean_text(text):
    return re.sub(r"\s+", " ", (text or "")).strip()


def _to_feature_key(text):
    # Normalize Azerbaijani chars for stable column names
    txt = _clean_text(text).lower()
    translit = {
        "ə": "e", "ö": "o", "ğ": "g", "ü": "u", "ı": "i", "ş": "s", "ç": "c",
        "Ə": "e", "Ö": "o", "Ğ": "g", "Ü": "u", "İ": "i", "Ş": "s", "Ç": "c",
    }
    for k, v in translit.items():
        txt = txt.replace(k, v)
    txt = txt.replace("/", " ")
    txt = re.sub(r"[^a-z0-9\s]", "", txt)
    txt = re.sub(r"\s+", "_", txt).strip("_")
    return txt


def _extract_number(text):
    if not text:
        return None
    m = re.search(r"(\d+(?:[\.,]\d+)?)", text)
    if not m:
        return None
    n = m.group(1).replace(",", ".")
    return float(n) if "." in n else int(n)


def _extract_base_fields(bs):
    rec = {
        "price": None,
        "rooms": None,
        "area_m2": None,
        "land_area_sot": None,
        "floor": None,
        "has_document": None,
        "address": None,
    }

    # Price
    price_box = bs.select_one("div.elan_new_price_box p")
    if price_box:
        ptxt = _clean_text(price_box.get_text(" ", strip=True))
        m = re.search(r"(\d[\d\s]*)", ptxt)
        if m:
            rec["price"] = int(m.group(1).replace(" ", ""))

    # Main attributes list
    main_items = [_clean_text(li.get_text(" ", strip=True)) for li in bs.select("ul.full.elan_property_list li")]
    for item in main_items:
        low = item.lower()
        low_compact = re.sub(r"\s+", "", low.replace("\xa0", " "))

        if rec["rooms"] is None and "otaq" in low:
            rec["rooms"] = _extract_number(low)

        if rec["area_m2"] is None and (
            "m²" in low or "m2" in low_compact or "kv.m" in low or "kvm" in low_compact
            or "sahə" in low or "sahe" in low
        ):
            rec["area_m2"] = _extract_number(low)
            if rec["area_m2"] is None:
                m_area = re.search(r"(\d+(?:[\.,]\d+)?)\s*(?:m²|m2|kv\.?m)", low)
                if m_area:
                    n = m_area.group(1).replace(",", ".")
                    rec["area_m2"] = float(n) if "." in n else int(n)

        if rec["land_area_sot"] is None and "sot" in low:
            rec["land_area_sot"] = _extract_number(low)

        if rec["floor"] is None and ("mərtəb" in low or "merteb" in low):
            rec["floor"] = _extract_number(low)

        if any(x in low for x in ["kupça", "kupca", "çıxarış", "cixaris"]):
            rec["has_document"] = "Yes"

    if rec["has_document"] is None:
        rec["has_document"] = "No"

    # Address
    address_parts = [_clean_text(li.get_text(" ", strip=True)) for li in bs.select("ul.elan_adr_list.full li")]
    rec["address"] = ", ".join([x for x in address_parts if x]) if address_parts else None

    return rec

In [ ]:
# Determine which column contains links
link_col = "url" if "url" in df_links.columns else ("link" if "link" in df_links.columns else df_links.columns[0])
urls = df_links[link_col].dropna().astype(str).tolist()[:1000]  # First 50 links only

all_records = []
all_features = []  # ordered global list of discovered dynamic features

for i, url in enumerate(tqdm(urls), 1):
    driver.get(url)
    time.sleep(1.5)
    bs = BeautifulSoup(driver.page_source, "html.parser")

    rec = {"link": url}
    rec.update(_extract_base_fields(bs))

    # Additional features section
    # If section missing => all known features should be No
    feature_lis = bs.select("ul.full.property_lists li")
    current_features = set()

    for li in feature_lis:
        ftxt = _clean_text(li.get_text(" ", strip=True))
        if not ftxt:
            continue
        fkey = _to_feature_key(ftxt)
        if not fkey:
            continue

        current_features.add(fkey)

        # New feature found: add globally + backfill old records with No
        if fkey not in all_features:
            all_features.append(fkey)
            for prev in all_records:
                prev[fkey] = "No"

    # Mark Yes/No for ALL known features
    for f in all_features:
        rec[f] = "Yes" if f in current_features else "No"

    all_records.append(rec)

# Final DataFrame
df_properties_dynamic = pd.DataFrame(all_records)

# Keep base columns first, then dynamic feature columns
base_cols = ["link", "price", "rooms", "area_m2", "land_area_sot", "floor", "has_document", "address"]
feature_cols = [c for c in all_features if c in df_properties_dynamic.columns]
ordered_cols = [c for c in base_cols if c in df_properties_dynamic.columns] + feature_cols

df_properties_dynamic = df_properties_dynamic[ordered_cols]

print(f"Done. Listings scraped: {len(df_properties_dynamic)}")
print(f"Dynamic features discovered: {len(feature_cols)}")
print(f"\nFeatures found: {feature_cols}")
print(f"\nFirst 5 listings:")
df_properties_dynamic.head()

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [1:08:49<00:00,  4.13s/it]

Done. Listings scraped: 1000
Dynamic features discovered: 19

Features found: ['temirli', 'qaz', 'su', 'isiq', 'avtodayanacaq', 'telefon', 'internet', 'pvc_pencere', 'balkon', 'kabel_tv', 'lift', 'kombi', 'metbex_mebeli', 'temirsiz', 'merkezi_qizdirici_sistem', 'kondisioner', 'esyali', 'hovuz', 'duzelme']

First 5 listings:


,link,price,rooms,area_m2,land_area_sot,floor,has_document,address,temirli,qaz,...,kabel_tv,lift,kombi,metbex_mebeli,temirsiz,merkezi_qizdirici_sistem,kondisioner,esyali,hovuz,duzelme
0,https://arenda.az/satilir-4-otaqli-heyet-evi-v...,115000.0,4.0,130.0,1.3,2.0,Yes,"Bakı, Abşeron, Masazır",Yes,Yes,...,No,No,No,No,No,No,No,No,No,No
1,https://arenda.az/satilir-3-otaqli-yeni-tikili...,195500.0,3.0,86.0,NaN,6.0,Yes,"Bakı, Xətai, Köhnə Günəşli qəs., metro Həzi As...",Yes,Yes,...,Yes,Yes,Yes,Yes,No,No,No,No,No,No
2,https://arenda.az/satilir-2-otaqli-yeni-tikili...,55000.0,2.0,78.0,NaN,4.0,Yes,"Bakı, Yasamal, Yeni Yasamal qəs., metro İnşaat...",No,No,...,No,No,No,No,No,No,No,No,No,No
3,https://arenda.az/satilir-7-otaqli-bag-evi-bin...,200000.0,7.0,NaN,4.0,2.0,No,"Bakı, Binəqədi, Biləcəri qəs., metro Avtovağzal",No,No,...,No,No,No,No,No,No,No,No,No,No
4,https://arenda.az/satilir-3-otaqli-yeni-tikili...,97000.0,3.0,92.0,NaN,8.0,No,"Bakı, Abşeron, Masazır",No,No,...,No,No,No,No,No,No,No,No,No,No


In [29]:
df_properties_dynamic.to_csv("satilir_properties_dynamic1.csv", index=False)

## 4) Large-Range Scraping + Image Download
This step scrapes listings from index **5500 to the end**, extracts dynamic features, and downloads gallery images into local folders.

Because of repeated crashes during long runs, the full scraping job was split into batches. The range from index **0 to 5499** was completed earlier, and this section processes only the remaining listings.

### Outputs
- `satilir_properties_5500_end_with_photos.csv`
- `satilir_photos_5500_end/` (downloaded images per listing folder)

In [ ]:
# NEW CODE: links 5500 to end + dynamic features + save photos

# --- make this cell self-contained (in case helper cell wasn't run) ---
if "_clean_text" not in globals():
    def _clean_text(text):
        return re.sub(r"\s+", " ", (text or "")).strip()

if "_to_feature_key" not in globals():
    def _to_feature_key(text):
        txt = _clean_text(text).lower()
        translit = {
            "ə": "e", "ö": "o", "ğ": "g", "ü": "u", "ı": "i", "ş": "s", "ç": "c",
            "Ə": "e", "Ö": "o", "Ğ": "g", "Ü": "u", "İ": "i", "Ş": "s", "Ç": "c",
        }
        for k, v in translit.items():
            txt = txt.replace(k, v)
        txt = txt.replace("/", " ")
        txt = re.sub(r"[^a-z0-9\s]", "", txt)
        txt = re.sub(r"\s+", "_", txt).strip("_")
        return txt

if "_extract_number" not in globals():
    def _extract_number(text):
        if not text:
            return None
        m = re.search(r"(\d+(?:[\.,]\d+)?)", text)
        if not m:
            return None
        n = m.group(1).replace(",", ".")
        return float(n) if "." in n else int(n)

if "_extract_base_fields" not in globals():
    def _extract_base_fields(bs):
        rec = {
            "price": None,
            "rooms": None,
            "area_m2": None,
            "land_area_sot": None,
            "floor": None,
            "has_document": None,
            "address": None,
        }

        price_box = bs.select_one("div.elan_new_price_box p")
        if price_box:
            ptxt = _clean_text(price_box.get_text(" ", strip=True))
            m = re.search(r"(\d[\d\s]*)", ptxt)
            if m:
                rec["price"] = int(m.group(1).replace(" ", ""))

        main_items = [_clean_text(li.get_text(" ", strip=True)) for li in bs.select("ul.full.elan_property_list li")]
        for item in main_items:
            low = item.lower()
            low_compact = re.sub(r"\s+", "", low.replace("\xa0", " "))

            if rec["rooms"] is None and "otaq" in low:
                rec["rooms"] = _extract_number(low)

            if rec["area_m2"] is None and (
                "m²" in low or "m2" in low_compact or "kv.m" in low or "kvm" in low_compact
                or "sahə" in low or "sahe" in low
            ):
                rec["area_m2"] = _extract_number(low)
                if rec["area_m2"] is None:
                    m_area = re.search(r"(\d+(?:[\.,]\d+)?)\s*(?:m²|m2|kv\.?m)", low)
                    if m_area:
                        n = m_area.group(1).replace(",", ".")
                        rec["area_m2"] = float(n) if "." in n else int(n)

            if rec["land_area_sot"] is None and "sot" in low:
                rec["land_area_sot"] = _extract_number(low)

            if rec["floor"] is None and ("mərtəb" in low or "merteb" in low):
                rec["floor"] = _extract_number(low)

            if any(x in low for x in ["kupça", "kupca", "çıxarış", "cixaris"]):
                rec["has_document"] = "Yes"

        if rec["has_document"] is None:
            rec["has_document"] = "No"

        address_parts = [_clean_text(li.get_text(" ", strip=True)) for li in bs.select("ul.elan_adr_list.full li")]
        rec["address"] = ", ".join([x for x in address_parts if x]) if address_parts else None

        return rec


def _extract_gallery_urls(bs):
    urls = []
    gallery = bs.select_one("div.property-gallery")
    if not gallery:
        return urls

    for a in gallery.select('a[data-fancybox="gallery"][href]'):
        href = (a.get("href") or "").strip()
        if not href:
            continue
        if href.startswith("//"):
            href = "https:" + href
        elif href.startswith("/"):
            href = "https://arenda.az" + href
        if href.startswith("http") and href not in urls:
            urls.append(href)

    return urls


def _download_images(image_urls, listing_dir):
    listing_dir.mkdir(parents=True, exist_ok=True)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36",
        "Referer": "https://arenda.az/"
    }

    saved_paths = []
    for j, img_url in enumerate(image_urls, 1):
        try:
            parsed = urlparse(img_url)
            ext = os.path.splitext(parsed.path)[1].lower()
            if ext not in [".jpg", ".jpeg", ".png", ".webp", ".gif"]:
                ext = ".jpg"

            file_path = listing_dir / f"img_{j:03d}{ext}"
            r = requests.get(img_url, headers=headers, timeout=30)
            if r.status_code == 200 and r.content:
                with open(file_path, "wb") as f:
                    f.write(r.content)
                saved_paths.append(str(file_path))
        except Exception:
            continue

    return saved_paths


# links 5500th to end (1-based indexing)
start_idx = 5499

link_col = "url" if "url" in df_links.columns else ("link" if "link" in df_links.columns else df_links.columns[0])
urls = df_links[link_col].dropna().astype(str).tolist()[start_idx:]

photos_root = Path("satilir_photos_5500_end")
photos_root.mkdir(parents=True, exist_ok=True)

all_records = []
all_features = []

for i, url in enumerate(tqdm(urls, desc="Scraping links 5500-end"), start_idx + 1):
    try:
        driver.get(url)
        time.sleep(1.5)
        bs = BeautifulSoup(driver.page_source, "html.parser")

        rec = {"link": url}
        rec.update(_extract_base_fields(bs))

        # dynamic features yes/no
        feature_lis = bs.select("ul.full.property_lists li")
        current_features = set()

        for li in feature_lis:
            ftxt = _clean_text(li.get_text(" ", strip=True))
            if not ftxt:
                continue
            fkey = _to_feature_key(ftxt)
            if not fkey:
                continue

            current_features.add(fkey)
            if fkey not in all_features:
                all_features.append(fkey)
                for prev in all_records:
                    prev[fkey] = "No"

        for f in all_features:
            rec[f] = "Yes" if f in current_features else "No"

        # photos
        image_urls = _extract_gallery_urls(bs)
        listing_slug = re.sub(r"[^a-zA-Z0-9_-]", "_", url.rstrip("/").split("/")[-1])
        listing_dir = photos_root / f"{i:04d}_{listing_slug}"
        saved_paths = _download_images(image_urls, listing_dir)

        rec["photo_count"] = len(saved_paths)
        rec["photo_files"] = " | ".join(saved_paths) if saved_paths else None
        rec["photo_urls"] = " | ".join(image_urls) if image_urls else None

        all_records.append(rec)

    except Exception as e:
        rec = {
            "link": url,
            "price": None,
            "rooms": None,
            "area_m2": None,
            "land_area_sot": None,
            "floor": None,
            "has_document": None,
            "address": None,
            "photo_count": 0,
            "photo_files": None,
            "photo_urls": None,
            "error": str(e)
        }
        for f in all_features:
            rec[f] = "No"
        all_records.append(rec)

# final dataframe
base_cols = [
    "link", "price", "rooms", "area_m2", "land_area_sot", "floor",
    "has_document", "address", "photo_count", "photo_files", "photo_urls"
]

df_properties_with_photos_5500_end = pd.DataFrame(all_records)
feature_cols = [c for c in all_features if c in df_properties_with_photos_5500_end.columns]
other_cols = [c for c in df_properties_with_photos_5500_end.columns if c not in base_cols + feature_cols]
ordered_cols = [c for c in base_cols if c in df_properties_with_photos_5500_end.columns] + feature_cols + other_cols

df_properties_with_photos_5500_end = df_properties_with_photos_5500_end[ordered_cols]

df_properties_with_photos_5500_end.to_csv("satilir_properties_5500_end_with_photos.csv", index=False)

print(f"Done. Listings scraped: {len(df_properties_with_photos_5500_end)}")
print(f"Dynamic features discovered: {len(feature_cols)}")
print(f"Photos downloaded: {int(df_properties_with_photos_5500_end['photo_count'].fillna(0).sum())}")
print("CSV saved: satilir_properties_5500_end_with_photos.csv")
print("Images folder: satilir_photos_5500_end/")

df_properties_with_photos_5500_end.head()

Scraping links 5500-end: 100%|██████████| 798/798 [59:12<00:00,  4.45s/it]  

Done. Listings scraped: 798
Dynamic features discovered: 19
Photos downloaded: 9944
CSV saved: satilir_properties_5500_end_with_photos.csv
Images folder: satilir_photos_5500_end/


,link,price,rooms,area_m2,land_area_sot,floor,has_document,address,photo_count,photo_files,...,metbex_mebeli,esyali,avtodayanacaq,temirsiz,lift,kombi,duzelme,merkezi_qizdirici_sistem,hovuz,telefon
0,https://arenda.az/satilir-2-otaqli-kohne-tikil...,127500.0,2.0,50.0,NaN,2.0,Yes,"Bakı, Nəsimi, 5-ci mikrorayon, metro Memar Əcəmi",15,satilir_photos_5500_end/5500_satilir-2-otaqli-...,...,Yes,Yes,Yes,No,No,No,No,No,No,No
1,https://arenda.az/satilir-2-otaqli-yeni-tikili...,10000.0,2.0,50.0,NaN,1.0,Yes,"Bakı, Abşeron, Masazır",4,satilir_photos_5500_end/5501_satilir-2-otaqli-...,...,Yes,No,Yes,No,No,No,No,No,No,No
2,https://arenda.az/satilir-2-otaqli-yeni-tikili...,NaN,NaN,NaN,NaN,NaN,No,None,0,None,...,No,No,No,No,No,No,No,No,No,No
3,https://arenda.az/satilir-2-otaqli-yeni-tikili...,10000.0,2.0,50.0,NaN,1.0,Yes,"Bakı, Abşeron, Masazır",5,satilir_photos_5500_end/5503_satilir-2-otaqli-...,...,Yes,No,Yes,No,No,No,No,No,No,No
4,https://arenda.az/satilir-2-otaqli-yeni-tikili...,163000.0,2.0,55.0,NaN,3.0,Yes,"Bakı, Binəqədi, 9-cu mikrorayon, metro Memar Ə...",15,satilir_photos_5500_end/5504_satilir-2-otaqli-...,...,No,No,No,No,No,No,No,No,No,No


## 5) Dataset Consolidation
Concatenate batch CSV files, align columns, and export one unified dataset.

**Output:** `satilir_properties.csv`

In [ ]:
# Concatenate "(Copy)" CSV files in the required order and save as satilir_properties.csv

# Required file order (prefix-based)
required_prefix_order = [
    "satilir_properties_1000",
    "satilir_properties_1000_2000",
    "satilir_properties_2000_3000",
    "satilir_properties_3000_4500",
    "satilir_properties_4500_5500",
    "satilir_properties_5500_end",
]

# Collect only CSV files that contain "(Copy)" in filename
all_csvs = [p for p in Path('.').glob('*.csv') if '(Copy)' in p.name]

def _order_key(path_obj):
    name = path_obj.stem  # filename without .csv
    # Primary ordering by required prefix sequence
    for idx, pref in enumerate(required_prefix_order):
        if name.startswith(pref):
            return (idx, name.lower())
    # Non-matching files go to end
    return (999, name.lower())

# Keep only files that match required prefixes
selected_files = [
    p for p in all_csvs
    if any(p.stem.startswith(pref) for pref in required_prefix_order)
]
selected_files = sorted(selected_files, key=_order_key)

if not selected_files:
    print('No matching "(Copy)" files found for required satilir_properties_* pattern.')
else:
    print('Files to concatenate (in order):')
    for i, f in enumerate(selected_files, 1):
        print(f'{i}. {f.name}')

    dfs = [pd.read_csv(f, low_memory=False) for f in selected_files]
    merged = pd.concat(dfs, ignore_index=True, sort=False)

    # Reorder columns to fix mixed feature order across files
    base_cols = [
        'link', 'price', 'rooms', 'area_m2', 'land_area_sot', 'floor',
        'has_document', 'address', 'photo_count', 'photo_files', 'photo_urls'
    ]
    trailing_known = ['error']

    existing_base = [c for c in base_cols if c in merged.columns]
    existing_trailing = [c for c in trailing_known if c in merged.columns]

    # Everything else treated as dynamic feature columns and sorted for consistency
    feature_cols = [
        c for c in merged.columns
        if c not in set(existing_base + existing_trailing)
    ]
    feature_cols = sorted(feature_cols)

    final_cols = existing_base + feature_cols + existing_trailing
    merged = merged[final_cols]

    output_file = 'satilir_properties.csv'
    merged.to_csv(output_file, index=False)

    print(f'\nDone. Rows: {len(merged)}')
    print(f'Columns: {len(merged.columns)}')
    print(f'Saved: {output_file}')
    merged.head()

Files to concatenate (in order):
1. satilir_properties_1000_2000_with_photos (Copy).csv
2. satilir_properties_1000_with_photos (Copy).csv
3. satilir_properties_2000_3000_with_photos (Copy).csv
4. satilir_properties_3000_4500_with_photos (Copy).csv
5. satilir_properties_4500_5500_with_photos (Copy).csv
6. satilir_properties_5500_end_with_photos (Copy).csv

Done. Rows: 6260
Columns: 28
Saved: satilir_properties.csv


In [31]:
df_final = pd.read_csv('satilir_properties.csv', low_memory=False)

## 6) Data Quality Checks
Validate data quality before modeling:
- remove rows with missing `price`
- remove duplicates
- save cleaned table back to disk

In [20]:
df_final['price'].isna().sum()

75

In [21]:
if 'df_final' not in globals():
    df_final = pd.read_csv('satilir_properties.csv', low_memory=False)

before_rows = len(df_final)
df_final = df_final[df_final['price'].notna()].copy()
after_rows = len(df_final)
deleted_rows = before_rows - after_rows

df_final.to_csv('satilir_properties.csv', index=False)

In [22]:
df_final.duplicated().sum()

2

In [26]:
df_final[df_final.duplicated(keep=False)]

,link,price,rooms,area_m2,land_area_sot,floor,has_document,address,avtodayanacaq,balkon,...,lift,merkezi_qizdirici_sistem,metbex_mebeli,pvc_pencere,qaz,su,telefon,temirli,temirsiz,error
999,https://arenda.az/satilir-8-otaqli-heyet-evi-v...,450000.0,8.0,250.0,7.0,2.0,Yes,"Bakı, Sabunçu, Yeni Ramana",Yes,Yes,...,No,No,Yes,Yes,Yes,Yes,No,Yes,No,NaN
1961,https://arenda.az/satilir-8-otaqli-heyet-evi-v...,450000.0,8.0,250.0,7.0,2.0,Yes,"Bakı, Sabunçu, Yeni Ramana",Yes,Yes,...,No,No,Yes,Yes,Yes,Yes,No,Yes,No,NaN
5461,https://arenda.az/satilir-2-otaqli-kohne-tikil...,127500.0,2.0,50.0,NaN,2.0,Yes,"Bakı, Nəsimi, 5-ci mikrorayon, metro Memar Əcəmi",Yes,Yes,...,No,No,Yes,Yes,Yes,Yes,No,Yes,No,NaN
5462,https://arenda.az/satilir-2-otaqli-kohne-tikil...,127500.0,2.0,50.0,NaN,2.0,Yes,"Bakı, Nəsimi, 5-ci mikrorayon, metro Memar Əcəmi",Yes,Yes,...,No,No,Yes,Yes,Yes,Yes,No,Yes,No,NaN


In [27]:
df_final = df_final.drop_duplicates()

In [28]:
df_final.to_csv('satilir_properties.csv', index=False)

In [32]:
df_final.isnull().sum()

link                           0
price                          0
rooms                          2
area_m2                      260
land_area_sot               4987
floor                          2
has_document                   0
address                        0
avtodayanacaq                  0
balkon                         0
duzelme                        0
esyali                         0
hovuz                          0
internet                       0
isiq                           0
kabel_tv                       0
kombi                          0
kondisioner                    0
lift                           0
merkezi_qizdirici_sistem       0
metbex_mebeli                  0
pvc_pencere                    0
qaz                            0
su                             0
telefon                        0
temirli                        0
temirsiz                       0
dtype: int64

## 7) Image Folder to Price Mapping
Match image folder names to listing slugs and generate a consolidated ID-price mapping table.

**Output:** `satilir_id_price_folder_all.csv`

In [ ]:
base_dir = Path('/home/admin123/AI_project')

# Map each photos folder to its corresponding CSV
folder_csv_mapping = {
    'satilir_photos_1000': 'satilir_properties_1000_with_photos.csv',
    'satilir_photos_1000_2000': 'satilir_properties_1000_2000_with_photos.csv',
    'satilir_photos_2000_3000': 'satilir_properties_2000_3000_with_photos.csv',
    'satilir_photos_3000_4500': 'satilir_properties_3000_4500_with_photos.csv',
    'satilir_photos_4500_5500': 'satilir_properties_4500_5500_with_photos.csv',
    'satilir_photos_5500_end': 'satilir_properties_5500_end_with_photos.csv',
}

all_results = []

# Process each photo folder
for photos_folder, csv_filename in folder_csv_mapping.items():
    photos_dir = base_dir / photos_folder
    csv_path = base_dir / csv_filename
    
    if not photos_dir.exists():
        print(f"⊘ {photos_folder} not found, skipping...")
        continue
    
    if not csv_path.exists():
        print(f"⊘ {csv_filename} not found, skipping...")
        continue
    
    print(f"Processing {photos_folder}...")
    
    # Read CSV
    df = pd.read_csv(csv_path)
    df['price'] = pd.to_numeric(df['price'], errors='coerce')
    
    # Build slug -> price mapping
    df['slug'] = (
        df['link']
        .astype(str)
        .str.strip()
        .str.rstrip('/')
        .str.split('/')
        .str[-1]
    )
    slug_to_price = (
        df[['slug', 'price']]
        .dropna(subset=['slug'])
        .drop_duplicates(subset=['slug'], keep='first')
        .set_index('slug')['price']
    )
    
    # Process subfolders in this photo directory
    folder_count = 0
    for subfolder in sorted(photos_dir.iterdir()):
        if not subfolder.is_dir():
            continue
        
        # Extract ID and slug from folder name
        # Pattern: 001_satilir-..., 0602_satilir-..., etc.
        m = re.match(r'^(\d+)_([^/]+)$', subfolder.name)
        if not m:
            continue
        
        house_id_str, slug = m.groups()
        house_id = int(house_id_str)
        price = slug_to_price.get(slug, pd.NA)
        
        all_results.append({
            'house_id': house_id,
            'price': price,
            'image_folder_name': subfolder.name,
            'source_folder': photos_folder,
        })
        folder_count += 1
    
    print(f"  ✓ Found {folder_count} image folders")

# Create consolidated DataFrame
result_df = pd.DataFrame(all_results).sort_values('house_id').reset_index(drop=True)

# Save
output_path = base_dir / 'satilir_id_price_folder_all.csv'
result_df.to_csv(output_path, index=False)

print(f"\n✓ Created: {output_path}")
print(f"✓ Total rows: {len(result_df)}")
print(f"✓ Price range: {result_df['price'].min()} - {result_df['price'].max()}")
print(f"\nPreview:")
print(result_df.head(15))

⊘ satilir_properties_1000_with_photos.csv not found, skipping...
Processing satilir_photos_1000_2000...
  ✓ Found 1000 image folders
Processing satilir_photos_2000_3000...
  ✓ Found 1001 image folders
Processing satilir_photos_3000_4500...
  ✓ Found 1474 image folders
Processing satilir_photos_4500_5500...
  ✓ Found 995 image folders
Processing satilir_photos_5500_end...
  ✓ Found 798 image folders

✓ Created: /home/admin123/AI_project/satilir_id_price_folder_all.csv
✓ Total rows: 5268
✓ Price range: 7300.0 - 63000000.0

Preview:
    house_id     price                                  image_folder_name  \
0       1001  200000.0  1001_satilir-3-otaqli-yeni-tikili-nerimanov-ra...   
1       1002  800000.0  1002_satilir-4-otaqli-yeni-tikili-nerimanov-ra...   
2       1003  210000.0  1003_satilir-2-otaqli-yeni-tikili-nerimanov-ra...   
3       1004       NaN                    1004_neftcilerde-obyekt-satilir   
4       1005  149000.0  1005_satilir-4-otaqli-kohne-tikili-bineqedi-ra...   
5 

In [34]:
base_dir = Path('/home/admin123/AI_project')
file_path = base_dir / 'satilir_id_price_folder.csv'

df = pd.read_csv(file_path)

In [36]:
df_cleaned = df[df['price'].notna()].copy()

In [38]:
df_cleaned.to_csv(file_path, index=False)